# 3 — ECM–cell analysis

Linking cells to extracellular-matrix fibres and measuring how the two
are spatially coupled. Runs on synthetic data.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad

rng = np.random.default_rng(0)

def synthetic_tissue(n_per_niche=120, n_niches=3, seed=0):
    """Three spatially separated cell niches with two marker features."""
    rng = np.random.default_rng(seed)
    centres = np.array([[0.0, 0.0], [600.0, 0.0], [0.0, 600.0]])[:n_niches]
    coords, niche = [], []
    for i, c in enumerate(centres):
        coords.append(c + rng.normal(0, 60, (n_per_niche, 2)))
        niche += [f'niche_{i}'] * n_per_niche
    coords = np.vstack(coords)
    n = len(coords)
    obs = pd.DataFrame({
        'X_centroid': coords[:, 0],
        'Y_centroid': coords[:, 1],
        'imageid': 'demo_image',
        'label': np.arange(n),
        'niche': niche,
        'phenotype': rng.choice(['duct', 'immune', 'stroma'], n, p=[.4, .35, .25]),
        'area': rng.lognormal(4.6, 0.3, n),
        'nc_ratio': rng.uniform(0.15, 0.75, n),
        # a feature with real spatial structure, for the statistics below
        'gradient': coords[:, 0] + coords[:, 1] + rng.normal(0, 30, n),
        'noise': rng.normal(size=n),
    })
    adata = ad.AnnData(X=rng.lognormal(0, 1, (n, 2)))
    adata.var_names = ['CD8', 'Ki67']
    adata.obs = obs
    adata.obs_names = [f'cell_{i}' for i in range(n)]
    return adata

adata = synthetic_tissue()
adata

## A synthetic fibre field

In [ ]:
fibre_rng = np.random.default_rng(1)
n_fibres = 400
fx = fibre_rng.uniform(-150, 750, n_fibres)
fy = fibre_rng.uniform(-150, 750, n_fibres)
fibres = pd.DataFrame({
    'X_centroid': fx,
    'Y_centroid': fy,
    'imageid': 'demo_image',
    'fiber_type': fibre_rng.choice(['col1', 'col6'], n_fibres),
    # orientation is in DEGREES (skimage regionprops reports radians)
    'orientation': fibre_rng.uniform(0, 180, n_fibres),
    'alignment': fx + fy + fibre_rng.normal(0, 40, n_fibres),
})
fibres.head()

## Link cells to nearby fibres

`build_cell_fiber_links` returns every cell–fibre pair within a radius;
`build_nearest_cell_fiber_map` keeps only the closest fibre per cell.

In [ ]:
import spatioev as sv

links = sv.tl.build_cell_fiber_links(adata, fibres, radius=60)
print(f'{len(links)} links for {adata.n_obs} cells')
links.head()

In [ ]:
nearest = sv.tl.build_nearest_cell_fiber_map(adata, fibres)
nearest['distance'].describe()

## Fibre orientation as vectors

Angles become unit vectors. **Orientation is interpreted in degrees** —
convert with `np.degrees()` if your source reports radians.

In [ ]:
vec = sv.tl.fiber_vectors(fibres)
vec[['orientation', 'vx', 'vy']].head()

## Spatial structure of the matrix

Moran's I over fibres shows whether the `alignment` feature is
spatially organised. These run on a sparse weight matrix, so they scale
to large fibre fields.

In [ ]:
i = sv.tl.morans_i_fibers(fibres, feature='alignment', k=8)
print(f'Moran I (fibre alignment) = {float(i): .4f}')

## Coupling ECM to cells

Cross Moran's I measures whether a fibre feature and a cell feature
co-vary in space.

In [ ]:
cross = sv.tl.cross_morans_i_ecm_cells(
    adata, fibres, links, 'alignment', 'gradient', k=8
)
cross